# Running the whole network

`01_one_check.ipynb` drives one check by hand. Here the loop just runs — and
because results transfer upstream, watching the whole network is watching the
dependency ladder work: **terminal reaches build first; everything else waits
for its downstream neighbour.**

In this milestone only `build_model` exists, so the network settles at
*terminals finished, everyone else waiting on downstream ND* — a library no
job can produce yet. That standstill is the ladder holding, not a failure:
when `run_nd_scenarios` lands, the same loop, unchanged, will carry the wave
upstream.

Two modes, one switch:

| `RUN_FOREVER` | behaviour |
|---|---|
| `False` | stop at steady state: nothing in flight and a pass that submitted nothing |
| `True` | keep going the way a service would; interrupt the cell to stop |

Interrupting is always safe — every fact the loop needs is in the database
before the step that wrote it returns.

**Prerequisites:** stack up, `build_model` image available, and the network
seeded (`01_one_check.ipynb` section 1 does that).

In [ ]:
import os
import time

import pandas as pd

from recon import check, db, jobs, processing, queue
from recon.config import settings
from recon.workers import LocalDockerRunner


def build_runner():
    env_vars = {"AWS_ENDPOINT_URL": "http://minio:9000"}
    for key in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN", "AWS_REQUEST_PAYER"):
        if os.environ.get(key):
            env_vars[key] = os.environ[key]
    return LocalDockerRunner(
        image=settings.build_model_image, network=settings.docker_network, env_vars=env_vars,
        platform=settings.docker_platform,
        volumes=[f"{settings.docker_data_dir}:/data:ro"] if settings.docker_data_dir else [])


def tally():
    return db.one("""
        SELECT count(*) FILTER (WHERE state = 'finished')           AS finished,
               count(*) FILTER (WHERE state = 'waiting_downstream') AS waiting,
               count(*) FILTER (WHERE state = 'in_flight')          AS in_flight,
               count(*) FILTER (WHERE state = 'resting')            AS resting,
               count(*) FILTER (WHERE state = 'halted')             AS halted,
               count(*)                                             AS total
        FROM reach_status""")


runner = build_runner()
print(f"image {runner.image} on network {runner.network}")
print(tally())

## Settings

The cap is on **submissions**, not checks — checking is cheap and cannot start
work by itself, and checking in-flight reaches is how results get noticed. How
many jobs may run at once is still an open question in the design doc, so
until it is settled the number belongs to whoever runs the loop.

In [ ]:
RUN_FOREVER = False
MAX_IN_FLIGHT = 4
INTERVAL = 8

## The loop

One pass is two questions put to the database, in order:

1. **which jobs are we waiting on?** — poll them; clear markers for the
   finished; request checks. First, so freed capacity is usable this pass.
2. **which reaches are due?** — check each: observe, gap, act.

Neither question needs anything remembered from the previous pass.

In [ ]:
started = time.time()
passes = 0
try:
    while True:
        passes += 1
        for outcome in jobs.status_pass(runner):
            if outcome["status"] in ("succeeded", "failed"):
                print(f"    reach {outcome['reach_id']} {outcome['step']}: "
                      f"{outcome['status']} - {outcome['action']}")

        submitted = 0
        for row in queue.due_reaches():
            if len(processing.in_flight()) >= MAX_IN_FLIGHT:
                break
            if check.run_check(row["reach_id"], runner).submitted_ref:
                submitted += 1

        now = tally()
        print(f"[{time.time() - started:5.0f}s] pass {passes:3d}  "
              f"finished {now['finished']:>3}/{now['total']}  waiting {now['waiting']:>3}  "
              f"in flight {now['in_flight']:>2}  resting {now['resting']:>2}  "
              f"halted {now['halted']:>2}  submitted {submitted}")

        steady = now["in_flight"] == 0 and submitted == 0 and passes > 1
        if steady and not RUN_FOREVER:
            print(f"\nsteady after {passes} passes in {time.time() - started:.0f}s")
            break
        time.sleep(INTERVAL)
except KeyboardInterrupt:
    print("\nstopped by hand; nothing lost — in-flight jobs are recorded in the database")

## Where the network stands

`finished` and `waiting_downstream` together should cover the network:
terminals done, everything else blocked on a downstream ND library that the
next milestone's job will produce.

In [ ]:
print(pd.DataFrame(db.query(
    "SELECT state, count(*) AS reaches FROM reach_status GROUP BY state ORDER BY reaches DESC")))
print()
print(db.one("""SELECT count(*) FILTER (WHERE rn.is_terminal) AS terminals,
                      count(*) FILTER (WHERE rs.state='finished') AS finished
               FROM reach_status rs JOIN reach_network rn USING (reach_id)"""))

## The wait graph

Every waiting reach points at the neighbour it waits for — recorded so a
viewer can draw this without recomputing any gap. Reaches whose downstream is
itself waiting form the chains that will drain upstream, one rung per ND
library, once that job exists.

In [ ]:
pd.DataFrame(db.query("""
    SELECT p.blocked_on_reach_id                    AS waits_on,
           ds.state                                 AS its_state,
           count(*)                                 AS reaches_waiting
    FROM reach_status rs
    JOIN reach_processing p USING (reach_id)
    JOIN reach_status ds ON ds.reach_id = p.blocked_on_reach_id
    WHERE rs.state = 'waiting_downstream'
    GROUP BY 1, 2 ORDER BY reaches_waiting DESC"""))

## Anything parked

In [ ]:
halted = db.query("SELECT reach_id, consecutive_failures, halted_at, last_error "
                  "FROM reach_processing WHERE halted")
if halted:
    display(pd.DataFrame(halted))
    print("\nafter fixing the cause:  processing.clear_halt(reach_id)")
else:
    print("nothing halted")